# 01 Parameter Sweep: Synthetic

This notebook keeps demand passive and sweeps only bandwidth target/limit settings. Elasticity, builder packing, and full state separation are intentionally out of scope for this milestone.

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim import generate_synthetic_blocks, load_config, replay
from sim.metrics import compute_metrics


In [ ]:
base_config = load_config(PROJECT_ROOT / "configs" / "synthetic_bandwidth_only.yaml")
blocks = generate_synthetic_blocks(base_config)


## Target / Limit Sweep

Try targets at 40%, 50%, 60%, and 70% of each candidate bandwidth limit.

In [ ]:
target_ratios = [0.4, 0.5, 0.6, 0.7]
limit_bytes_values = [1_200_000, 1_500_000, 2_000_000]

sweep_rows = []
for limit_bytes in limit_bytes_values:
    for target_ratio in target_ratios:
        target_bytes = int(limit_bytes * target_ratio)
        config = replace(
            base_config,
            bandwidth=replace(
                base_config.bandwidth,
                target_bytes=target_bytes,
                limit_bytes=limit_bytes,
            ),
        )
        df = replay(blocks, config)
        row = compute_metrics(df, config).iloc[0].to_dict()
        row.update(
            {
                "target_ratio": target_ratio,
                "target_bytes": target_bytes,
                "limit_bytes": limit_bytes,
            }
        )
        sweep_rows.append(row)

sweep = pd.DataFrame(sweep_rows)
sweep


## Sweep Plots

In [ ]:
metrics_to_plot = [
    "bandwidth_base_fee_volatility",
    "bandwidth_limit_hit_frequency",
    "mean_bandwidth_usage_over_target",
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, metrics_to_plot):
    pivot = sweep.pivot(index="target_ratio", columns="limit_bytes", values=metric)
    pivot.plot(ax=ax, marker="o")
    ax.set_title(metric)
    ax.set_xlabel("target / limit")
    ax.legend(title="limit bytes")
fig.tight_layout()
